# IND320 – Project work, part 2: Data sources
**Course:** IND320 Data to decision (NMBU, autumn 2026)
**Student:** Zarar Zafar

## Links
- **GitHub repository:** [https://github.com/zararzafar/zarar](https://github.com/zararzafar/zarar) (work for this part is on the branch `part2`)
- **Streamlit app:** [https://zarar-ind320.streamlit.app](https://zarar-ind320.streamlit.app)

## Contents
1. Work log
2. Setup (packages, tokens, Spark)
3. NVE Magasinstatistikk API (replaces the CSV file)
4. Part 1 plots recreated from the API
5. ENTSO-E cross-border flows
6. Storing the flows in Cassandra with Spark
7. Extracting data with Spark and plotting
8. Inserting the Spark-extracted data into MongoDB

## 1. Work log
*(300–500 words, describing both the Jupyter Notebook and the Streamlit work)*

In this part the project moved from a static CSV file to live data sources and databases. To avoid
mixing unfinished work with the version that is being peer reviewed, all changes were made on a new
GitHub branch called `part2`, which will be merged into `main` after the feedback round.

The first step was to replace `reservoirs.csv` with the NVE Magasinstatistikk API. The endpoint
`HentOffentligData` returns exactly the same columns as the CSV file, so the renaming from part 1
could be reused directly. The experience with the time columns from part 1 was useful: the rows are
still not sorted, so the data are sorted by area and date before plotting. The endpoint
`HentOmråder` describes the areas: the whole country (`NO`), the five price areas NO1–NO5 (`EL`) and
four water-course areas (`VASS`). The plots from part 1 were recreated from the API data.

The second data source was the ENTSO-E Transparency Platform. After registering and getting a
security token, the `entsoe-py` package was used to download cross-border physical flows for all
connections between the Norwegian bidding zones and the neighbouring zones in Sweden, Finland,
Denmark, Germany, the Netherlands and Great Britain for the last two years, in both directions. Some
connections report every 15 minutes and others every hour, so everything was resampled to hourly
mean power, which is the same as energy in MWh per hour.

The hourly flows were written to a local Cassandra database through Spark, using the
Spark–Cassandra connector from the course book. Spark was then used to read the table back, select
the relevant columns and aggregate the data: yearly totals for the pie chart, the first month of a
year for the line plot, and daily sums that were inserted into MongoDB. Storing daily values keeps the
MongoDB collection small, so the Streamlit app loads quickly.

In the Streamlit app the CSV file was removed. The reservoir pages now download from the NVE API, and
the month slider decides which period is downloaded. Radio buttons choose between `NO`, `EL` and
`VASS`, and both the plot and the `LineChartColumn` table show one line per area number. The
controls are in the sidebar and are remembered when switching between the table and the plot page. A
new page reads the flows from MongoDB and uses `st.columns`: radio buttons and a year selector with a
pie chart on the left, and `st.pills` for choosing months with a line plot on the right. An expander
documents the data sources. The MongoDB connection string is stored in Streamlit secrets, locally in
`.streamlit/secrets.toml` (ignored by git) and in the app settings on streamlit.app, so no password
is pushed to GitHub. The old page four was moved to page five.

The most time-consuming parts were getting the Spark–Cassandra connection to work and handling the
different time resolutions and time zones from ENTSO-E.

## 2. Setup
Packages used (install once with pip if needed):
`pip install requests pandas matplotlib entsoe-py pyspark cassandra-driver pymongo`

Tokens and passwords are **not** written in the notebook. They are read from environment variables
(or a local file that is listed in `.gitignore`):
- `ENTSOE_TOKEN` – security token from the ENTSO-E Transparency Platform
- `MONGO_URI` – connection string for MongoDB Atlas

In [ ]:
# Standard libraries and data handling
import os
import calendar
import requests
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Read secrets from environment variables. Alternatively, put them in a local file
# 'secrets_local.py' (listed in .gitignore) with ENTSOE_TOKEN = "..." and MONGO_URI = "...".
try:
    from secrets_local import ENTSOE_TOKEN, MONGO_URI
except ImportError:
    ENTSOE_TOKEN = os.environ.get("ENTSOE_TOKEN")
    MONGO_URI = os.environ.get("MONGO_URI")

print("ENTSO-E token found:", ENTSOE_TOKEN is not None)
print("MongoDB URI found:  ", MONGO_URI is not None)

## 3. NVE Magasinstatistikk API
The API is documented at <https://biapi.nve.no/magasinstatistikk/swagger>. Two endpoints are used:
- `HentOmråder` – the areas covered by the statistics
- `HentOffentligData` – all weekly observations (the same columns as the CSV file in part 1)

In [ ]:
# Base address of the NVE Magasinstatistikk API
NVE_URL = "https://biapi.nve.no/magasinstatistikk/api/Magasinstatistikk"

# Which areas does the API cover? The answer has three groups: land, elspot and vassdrag.
areas_json = requests.get(f"{NVE_URL}/HentOmråder", timeout=30).json()[0]
areas = pd.DataFrame([area for group in areas_json.values() for area in group])
areas[["omrType", "omrnr", "navn", "beskrivelse"]]

In [ ]:
# Download all weekly reservoir data from the API
response = requests.get(f"{NVE_URL}/HentOffentligData", timeout=60)
response.raise_for_status()          # stop with an error message if the request failed
raw = pd.DataFrame(response.json())

# Same English column names as in part 1
column_names = {
    "dato_Id": "date", "omrType": "area_type", "omrnr": "area_number",
    "iso_aar": "iso_year", "iso_uke": "iso_week", "fyllingsgrad": "fill_rate",
    "kapasitet_TWh": "capacity_TWh", "fylling_TWh": "stored_energy_TWh",
    "neste_Publiseringsdato": "next_publication_date",
    "fyllingsgrad_forrige_uke": "fill_rate_previous_week",
    "endring_fyllingsgrad": "fill_rate_change",
}
reservoirs = raw.rename(columns=column_names)

# Time columns: 'date' is the date of the weekly observation (a Sunday).
# Lesson from part 1: the rows are not sorted, so sort by area and date.
reservoirs["date"] = pd.to_datetime(reservoirs["date"])
reservoirs = reservoirs.sort_values(["area_type", "area_number", "date"]).reset_index(drop=True)

print("Rows:", len(reservoirs))
print("Period:", reservoirs["date"].min().date(), "to", reservoirs["date"].max().date())
print(reservoirs.groupby(["area_type", "area_number"]).size().rename("weeks").to_string())

## 4. Part 1 plots recreated from the API
As in part 1, the national total for Norway (`area_type == "NO"`) is used.

In [ ]:
# National series for Norway, indexed by date
norway = reservoirs[reservoirs["area_type"] == "NO"].set_index("date")
value_columns = ["fill_rate", "capacity_TWh", "stored_energy_TWh",
                 "fill_rate_previous_week", "fill_rate_change"]
labels = {
    "fill_rate": ("Fill rate", "Share of capacity (0-1)"),
    "capacity_TWh": ("Reservoir capacity", "TWh"),
    "stored_energy_TWh": ("Stored energy", "TWh"),
    "fill_rate_previous_week": ("Fill rate previous week", "Share of capacity (0-1)"),
    "fill_rate_change": ("Weekly change in fill rate", "Change (fraction points)"),
}

# Each column on its own axis
fig, axes = plt.subplots(len(value_columns), 1, figsize=(11, 13), sharex=True)
for ax, col in zip(axes, value_columns):
    ax.plot(norway.index, norway[col], linewidth=0.8)
    ax.set_title(labels[col][0])
    ax.set_ylabel(labels[col][1])
    ax.grid(alpha=0.3)
axes[-1].set_xlabel("Date")
fig.suptitle("Norwegian hydropower reservoirs from the NVE API (weekly)", fontsize=14)
fig.tight_layout()
plt.show()

In [ ]:
# All columns together: min-max scaling (constant columns are drawn at 0.5)
def min_max_scale(frame):
    value_range = frame.max() - frame.min()
    scaled = (frame - frame.min()) / value_range.replace(0, 1)
    scaled.loc[:, value_range == 0] = 0.5
    return scaled

recent = min_max_scale(norway[value_columns])
recent = recent[recent.index >= recent.index.max() - pd.DateOffset(years=3)]

fig, ax = plt.subplots(figsize=(11, 5))
for col in value_columns:
    ax.plot(recent.index, recent[col], label=labels[col][0], linewidth=1.3)
ax.set_title("All columns together, min-max scaled (last three years, NVE API)")
ax.set_xlabel("Date")
ax.set_ylabel("Scaled value (0 = min, 1 = max)")
ax.grid(alpha=0.3)
ax.legend(fontsize=9)
fig.tight_layout()
plt.show()

## 5. ENTSO-E cross-border flows
Cross-border *physical flows* (ENTSO-E document type A11) are downloaded with the `entsoe-py`
package for every connection between a Norwegian bidding zone and a neighbouring zone, in both
directions, for the last two years. `entsoe-py` splits long periods into one-year requests itself.

In [ ]:
from entsoe import EntsoePandasClient
from entsoe.exceptions import NoMatchingDataError

client = EntsoePandasClient(api_key=ENTSOE_TOKEN)

# Connections from Norwegian bidding zones to neighbouring zones (entsoe-py area codes)
CONNECTIONS = [
    ("NO_1", "SE_3"), ("NO_2", "DK_1"), ("NO_2", "NL"), ("NO_2", "DE_LU"), ("NO_2", "GB"),
    ("NO_3", "SE_2"), ("NO_4", "SE_1"), ("NO_4", "SE_2"), ("NO_4", "FI"),
]

# The last two years, as whole months
END = pd.Timestamp.now(tz="Europe/Oslo").normalize().replace(day=1)
START = END - pd.DateOffset(years=2)
print("Period:", START.date(), "to", END.date())

In [ ]:
def download_flow(area_from, area_to):
    """Hourly mean flow in MW from area_from to area_to (empty Series if no data)."""
    try:
        series = client.query_crossborder_flows(area_from, area_to, start=START, end=END)
    except NoMatchingDataError:
        return pd.Series(dtype=float)
    # Some connections report every 15 minutes: average to hourly values.
    # Mean power over one hour (MW) = energy in that hour (MWh).
    return series.resample("1h").mean()

frames = []
for no_area, neighbour in CONNECTIONS:
    name = f"{no_area.replace('_', '')}-{neighbour.replace('_', '')}"   # e.g. 'NO2-DK1'
    for direction, (a, b) in {"export": (no_area, neighbour), "import": (neighbour, no_area)}.items():
        s = download_flow(a, b)
        print(f"{name:10s} {direction:6s}: {len(s):6d} hours")
        if s.empty:
            continue
        frames.append(pd.DataFrame({
            "connection": name,
            "direction": direction,
            "from_area": a,
            "to_area": b,
            # Cassandra stores timestamps in UTC
            "time": s.index.tz_convert("UTC").tz_localize(None),
            "flow_mw": s.values,
        }))

flows = pd.concat(frames, ignore_index=True).dropna(subset=["flow_mw"])
print(flows.shape)
flows.head()

## 6. Storing the flows in Cassandra with Spark
The keyspace and table are created with the Cassandra Python driver. The partition key is
(connection, direction) and the rows are ordered by time within each partition.

In [ ]:
from cassandra.cluster import Cluster

cluster = Cluster(["localhost"], port=9042)
session = cluster.connect()
session.execute("""
    CREATE KEYSPACE IF NOT EXISTS ind320
    WITH replication = {'class': 'SimpleStrategy', 'replication_factor': 1}
""")
session.execute("""
    CREATE TABLE IF NOT EXISTS ind320.crossborder_flows (
        connection text, direction text, time timestamp,
        from_area text, to_area text, flow_mw double,
        PRIMARY KEY ((connection, direction), time)
    )
""")
print("Keyspace and table are ready")

In [ ]:
# Spark setup as in the course book (edit JAVA_HOME / HADOOP_HOME for your own computer if needed)
# os.environ["JAVA_HOME"] = "C:/Program Files/Java/jdk-17"          # example for Windows
# os.environ["HADOOP_HOME"] = "C:/Hadoop/hadoop-3.3.1"                # Windows only
os.environ["PYSPARK_PYTHON"] = "python"
os.environ["PYSPARK_DRIVER_PYTHON"] = "python"
os.environ["PYSPARK_HADOOP_VERSION"] = "without"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (SparkSession.builder.appName("IND320_part2")
         .config("spark.jars.packages", "com.datastax.spark:spark-cassandra-connector_2.12:3.5.1")
         .config("spark.cassandra.connection.host", "localhost")
         .config("spark.cassandra.connection.port", "9042")
         .config("spark.sql.extensions", "com.datastax.spark.connector.CassandraSparkExtensions")
         .config("spark.sql.catalog.mycatalog", "com.datastax.spark.connector.datasource.CassandraCatalog")
         .config("spark.sql.session.timeZone", "UTC")
         .getOrCreate())

# Write the pandas data to Cassandra through Spark (append = insert/overwrite rows with the same key)
spark.createDataFrame(flows).write.format("org.apache.spark.sql.cassandra") \
    .options(table="crossborder_flows", keyspace="ind320").mode("append").save()
print("Rows written:", len(flows))

## 7. Extracting data with Spark and plotting
Spark reads the table back from Cassandra and selects only the columns that are needed.

In [ ]:
# Read from Cassandra and keep the relevant columns
flows_spark = (spark.read.format("org.apache.spark.sql.cassandra")
               .options(table="crossborder_flows", keyspace="ind320").load()
               .select("connection", "direction", "time", "flow_mw"))

# Add local (Norwegian) date and year, used for grouping
flows_spark = (flows_spark
               .withColumn("local_time", F.from_utc_timestamp("time", "Europe/Oslo"))
               .withColumn("date", F.to_date("local_time"))
               .withColumn("year", F.year("local_time")))
flows_spark.createOrReplaceTempView("flows")
flows_spark.show(5)

In [ ]:
# Pie chart: total import and export for one connection in one year
CONNECTION = "NO2-DK1"     # chosen connection
YEAR = 2025                # chosen year (the only complete year in the two-year window)

totals = (flows_spark.filter((F.col("connection") == CONNECTION) & (F.col("year") == YEAR))
          .groupBy("direction").agg((F.sum("flow_mw") / 1000).alias("GWh"))
          .toPandas().set_index("direction")["GWh"])

fig, ax = plt.subplots(figsize=(6, 6))
ax.pie(totals, labels=[f"{d.capitalize()} ({v:,.0f} GWh)" for d, v in totals.items()],
       autopct="%1.0f%%", startangle=90,
       colors=["tab:orange" if d == "export" else "tab:blue" for d in totals.index])
ax.set_title(f"Cross-border flow {CONNECTION} in {YEAR}: export from and import to Norway")
plt.show()

In [ ]:
# Line plot: the first month of the year, one line per connection and direction
first_month = (flows_spark.filter((F.col("year") == YEAR) & (F.month("local_time") == 1))
               .select("connection", "direction", "local_time", "flow_mw")
               .toPandas())

connections = sorted(first_month["connection"].unique())
fig, axes = plt.subplots(len(connections), 1, figsize=(11, 2.2 * len(connections)),
                         sharex=True, squeeze=False)
axes = axes[:, 0]                  # one column of subplots
for ax, conn in zip(axes, connections):
    for direction, group in first_month[first_month["connection"] == conn].groupby("direction"):
        group = group.sort_values("local_time")
        ax.plot(group["local_time"], group["flow_mw"], linewidth=0.8, label=direction,
                color="tab:orange" if direction == "export" else "tab:blue")
    ax.set_title(conn, fontsize=10)
    ax.set_ylabel("MW")
    ax.grid(alpha=0.3)
axes[0].legend(loc="upper right")
axes[-1].set_xlabel("Date")
fig.suptitle(f"Hourly cross-border flows, January {YEAR}", fontsize=14)
fig.tight_layout()
plt.show()

## 8. Inserting the Spark-extracted data into MongoDB
Spark sums the hourly values to daily energy (MWh) per connection and direction. These daily
values are inserted into MongoDB (database `ind320`, collection `crossborder_daily`), which the
Streamlit app reads.

In [ ]:
# Daily energy per connection and direction, computed with Spark
daily = (flows_spark.groupBy("connection", "direction", "date")
         .agg(F.sum("flow_mw").alias("energy_mwh"))
         .orderBy("connection", "direction", "date")
         .toPandas())
daily["date"] = pd.to_datetime(daily["date"])      # MongoDB stores datetimes, not dates
print(daily.shape)
daily.head()

In [ ]:
import pymongo

mongo = pymongo.MongoClient(MONGO_URI)
collection = mongo["ind320"]["crossborder_daily"]

# Replace old data so the notebook can be re-run without creating duplicates
collection.delete_many({})
result = collection.insert_many(daily.to_dict("records"))
print("Documents inserted:", len(result.inserted_ids))

# Check: read one document back
collection.find_one({}, {"_id": 0})

## Next steps
In part 3 the work continues on the same branch structure, building on the API connections and the
MongoDB data used by the Streamlit app.